# TruthShield 00 — Setup, data download & HAVIC zero-shot baseline
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. Safe to re-run: every step skips work already saved on Drive.

What this notebook does:
1. Mounts Drive, clones HAVIC + TruthShield code
2. Downloads HiFi-AVDF (~6.5 GB) and HAVIC weights (~860 MB) to Drive **once**
3. Unzips videos to fast local disk (`/content/hifi`) each session
4. Builds leakage-free pair-level splits
5. Verifies the HAVIC checkpoint loads with **zero missing keys**
6. Runs the official HAVIC sliding-window inference on the test split → baseline row of the ablation table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/TruthShield'
for d in ['data/HiFi-AVDF', 'weights', 'splits', 'results', 'cache']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)

# TruthShield code (public GitHub repo - no token needed)
TS_REPO = 'https://github.com/Jawad374/TruthShield.git'
# (Set TS_REPO = '' to instead use a copy of the code in Drive at ROOT/code.)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
if not os.path.exists('/content/HAVIC'):
    !git clone -q --depth 1 https://github.com/tuffy-studio/HAVIC /content/HAVIC
# Patch HAVIC for current timm: new timm's Attention has no 'qk_scale' argument.
# HAVIC always passes qk_scale=None (= default scale), so dropping it changes nothing.
_f = '/content/HAVIC/src/models/interaction_modules.py'
_s = open(_f).read()
_s = _s.replace('qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)',
                'qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)')
open(_f, 'w').write(_s)
print('HAVIC timm patch applied:', 'qk_scale=qk_scale, attn_drop' not in _s)

if TS_REPO:
    if not os.path.exists('/content/TruthShield'):
        !git clone -q {TS_REPO} /content/TruthShield
    !git -C /content/TruthShield pull -q
    TS = '/content/TruthShield'
else:
    TS = f'{ROOT}/code'
assert os.path.exists(f'{TS}/src/models/dynamic_fusion.py'), f'TruthShield code not found at {TS}'

# Only what inference needs. Do NOT install HAVIC's requirements.txt (pins torch 2.3 / numpy 1.24 and breaks Colab).
!pip install -q decord ffmpeg-python einops timm huggingface_hub
# FaceX-Zoo torch.load()s whole pickled models; torch>=2.6 blocks that by default.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

## Download dataset + weights to Drive (one time)

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download('JielunPeng/HiFi-AVDF', repo_type='dataset', local_dir=f'{ROOT}/data/HiFi-AVDF')
W = hf_hub_download('JielunPeng/HAVIC', 'best_ft_model.pth', local_dir=f'{ROOT}/weights')
!ls -lh {ROOT}/data/HiFi-AVDF {ROOT}/weights

## Unzip to local disk (every session — reading video from Drive is slow)

In [ ]:
import glob, zipfile
LOCAL = '/content/hifi'
os.makedirs(LOCAL, exist_ok=True)
for z in sorted(glob.glob(f'{ROOT}/data/HiFi-AVDF/*.zip')):
    eng = os.path.basename(z)[:-4]
    if os.path.exists(f'{LOCAL}/.done_{eng}'):
        continue
    print('unzipping', eng)
    with zipfile.ZipFile(z) as f:
        f.extractall(LOCAL)
    open(f'{LOCAL}/.done_{eng}', 'w').close()

# Map CSV paths (e.g. 'sora2/1234.mp4') to real files, whatever the zip layout is.
mp4s = glob.glob(f'{LOCAL}/**/*.mp4', recursive=True)
by_rel = {}
for p in mp4s:
    parts = p.split('/')
    by_rel[parts[-2] + '/' + parts[-1]] = p
print(len(mp4s), 'videos on local disk')

## Leakage-free splits (pair-level, stratified by engine) + leave-one-engine-out

In [ ]:
import sys, pandas as pd

def use_src(root):
    """HAVIC and TruthShield both have a top-level `src` package.
    Drop any cached `src` modules and put `root` first so `import src...` resolves there."""
    for k in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
        del sys.modules[k]
    for p in ('/content/HAVIC', TS):
        while p in sys.path:
            sys.path.remove(p)
    sys.path.insert(0, root)

!cd {TS} && python -m src.data.make_splits --data_root {ROOT}/data/HiFi-AVDF --out_dir {ROOT}/splits
split = pd.read_csv(f'{ROOT}/splits/hifi_main.csv')
split['abs_path'] = split['video_path'].map(by_rel)
missing = split['abs_path'].isna().sum()
print('rows:', len(split), '| missing files:', missing)
assert missing == 0, split[split.abs_path.isna()].head()
split.to_csv(f'{ROOT}/splits/hifi_main_abs.csv', index=False)

## Verify the HAVIC checkpoint actually loads
The official script uses `strict=False`, which silently ignores mismatched keys (e.g. a `module.` prefix).
We check explicitly — a non-empty *missing* list means the baseline would be meaningless.

In [ ]:
import torch
use_src('/content/HAVIC')
from src.models.HAVIC import HAVIC_FT
sd = torch.load(W, map_location='cpu')
sd = sd.get('state_dict', sd) if isinstance(sd, dict) else sd
prefixes = {k.split('.')[0] for k in sd}
print('top-level key prefixes:', sorted(prefixes)[:15])
sd = {k.removeprefix('module.'): v for k, v in sd.items()}
m = HAVIC_FT()
missing, unexpected = m.load_state_dict(sd, strict=False)
print('missing:', len(missing), missing[:10])
print('unexpected:', len(unexpected), unexpected[:10])
assert not missing, 'HAVIC weights did not load cleanly'
del m, sd; torch.cuda.empty_cache()
use_src(TS)

## HAVIC zero-shot baseline (official sliding-window inference)
HAVIC was fine-tuned on FakeAVCeleb only, so this is a true zero-shot number on HiFi-AVDF.
`SUBSET='test'` (762 clips, ~20–40 min) gives the ablation-table baseline. `'all'` (3,810 clips) gives per-engine numbers comparable with the paper.

In [ ]:
SUBSET = 'test'   # 'test' or 'all'
sub = split if SUBSET == 'all' else split[split.split == 'test']
in_csv = f'/content/havic_in_{SUBSET}.csv'
sub[['abs_path', 'label']].rename(columns={'abs_path': 'video_path', 'label': 'overall_label'}).to_csv(in_csv, index=False)
out_csv = f'{ROOT}/results/havic_zeroshot_{SUBSET}.csv'

# The official loader prints missing keys; if the check above found a 'module.' prefix, save a cleaned copy.
W_CLEAN = f'{ROOT}/weights/best_ft_model_clean.pth'
if not os.path.exists(W_CLEAN):
    sd = torch.load(W, map_location='cpu'); sd = sd.get('state_dict', sd)
    torch.save({k.removeprefix('module.'): v for k, v in sd.items()}, W_CLEAN); del sd

%cd /content/HAVIC/evaluation
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python sliding_window_infer.py --csv_file_path {in_csv} --save_csv_path {out_csv} --finetune_path {W_CLEAN} --mode evaluation --max_workers 2 2>&1 | grep -vE "^\[Warning\]|Audio successfully" | tail -25
%cd /content

In [ ]:
use_src(TS)
from src.utils.metrics import compute_metrics, to_markdown
res = pd.read_csv(out_csv).merge(split[['abs_path', 'engine']], left_on='video_path', right_on='abs_path')
print(f'scored {len(res)}/{len(sub)} clips (the rest failed face detection / too short)')
rows = {'HAVIC zero-shot (all engines)': compute_metrics(res.overall_label, res.overall_pred)}
for eng, g in res.groupby('engine'):
    rows[f'  {eng}'] = compute_metrics(g.overall_label, g.overall_pred)
table = to_markdown(rows)
print(table)
open(f'{ROOT}/results/havic_zeroshot_{SUBSET}.md', 'w').write(table)